# Storing Chat History in 3rd Party Storage

### By default, when using `ChatAgent`, chat history is stored in memory in the `AgentThread` object or the underlying inference service, if the service supports it
### But we need persistence chat history, for which the external database is to be used.
### Here we will use Redis database to store the chat history, the in-memory behavior will not be possible if you want to buid any real production agents

## First Step is to install Redis

In [3]:
# ensure you have docker installed
# Run Redis official image using docker
!docker run -d --name my-redis -p 6379:6379 redis:latest

1a1576b5e02662a483103464e654583de1f702b8352b45fff420be7c65b68a63


In [4]:
# check if the container is Running
!docker ps
# also you I can connect to the container to test it
# docker exec -it my-redis redis-cli

CONTAINER ID   IMAGE          COMMAND                  CREATED        STATUS        PORTS                                         NAMES
1a1576b5e026   redis:latest   "docker-entrypoint.s…"   1 second ago   Up 1 second   0.0.0.0:6379->6379/tcp, [::]:6379->6379/tcp   my-redis


## Import Dependencies

Import the required libraries and Microsoft Agent Framework components:

- `asyncio`: For async/await support
- `os`: For accessing environment variables
- `json`: For JSON parsing
- `dotenv`: For loading environment variables from `.env` file
- `ChatAgent`: The main Agent class for building conversational AI agents
- `OpenAIChatClient`: Client for LLM inference using OpenAI-compatible endpoints (OpenRouter in this case)

In [2]:
import redis

# Connect to your Redis instance (use your actual host/port)
r = redis.Redis(host='localhost', port=6379, decode_responses=True)

# Delete the specific thread key (check your notebook for the exact key prefix)
# The default prefix is often "chat_messages:{thread_id}" or similar
r.delete("chat_messages:default_thread") # Replace with your actual key
print("Redis history cleared.")

Redis history cleared.


In [3]:
# Import core dependencies to create the agent, for Agent Framework
import asyncio
import os
import json

from dotenv import load_dotenv, find_dotenv
# Core components for building Agent, tool-enabled agents
from agent_framework import ChatAgent
from agent_framework.openai import OpenAIChatClient
from agent_framework import AgentThread
from agent_framework.redis import RedisChatMessageStore

## Load Environment Variables

Load environment variables from a `.env` file in the project directory. This file should contain:
- `OPENROUTER_ENDPOINT`: The OpenRouter API endpoint URL
- `OPENROUTER_API_KEY`: Your OpenRouter API key

In [4]:
# load environment file
load_dotenv(find_dotenv())

True

## Setup Chat Client

Configure the `OpenAIChatClient` to use OpenRouter API, which provides access to various LLM models including NVIDIA's Nemotron model. The client is configured with:

- `base_url`: The OpenRouter API endpoint
- `api_key`: Your API key for authentication
- `model_id`: The specific model to use (NVIDIA Nemotron 3 Nano 30B in this case)

In [5]:
# Setup OpenAIChatClient for LLM Inference - Here we will use OpenRouter API which is compatible with OpenAI and NVIDIA 30B model
# This client connects to the OpenRouter Models which are OpenAI-compatible endpoint
# Environment variables required
# OPENROUTER_ENDPOINT - 
# OPENROUTER_API_KEY
openai_chat_client = OpenAIChatClient(
    base_url=os.environ.get("GROQ_ENDPOINT"),
    api_key=os.environ.get("GROQ_API_KEY"),
    model_id="openai/gpt-oss-20b"
)

In [6]:
AGENT_NAME = "FoodAgent"

AGENT_INSTRUCTIONS = """You are an expert AI Chef dedicated to helping users discover and prepare delicious meals. Keep it concise, short and effective.
"""

## Create the Food Agent

Create the first agent (`food_agent`) with:

- `name`: "FoodAgent"
- `chat_client`: The OpenAI chat client configured earlier
- `instructions`: The behavior instructions defined above
- `chat_message_store`: External Database Store for persistent message.

## Basic example of using Redis chat message store.

In [7]:
 # Create Redis store with auto-generated thread ID
redis_store = RedisChatMessageStore(
        redis_url="redis://localhost:6379",
        # thread_id will be auto-generated if not provided
)

In [8]:
print(f"Created store with thread ID: {redis_store.thread_id}")

Created store with thread ID: thread_a15c54e0-cd36-4c09-9ae1-93ea0091cf95


In [9]:
thread = AgentThread(message_store=redis_store)

In [10]:
# create agent
# Here we create the foodAgent
# create the agent remember we are not using any tools here, this is simple example
food_agent = ChatAgent(
    name = AGENT_NAME,
    chat_client=openai_chat_client,
    instructions=AGENT_INSTRUCTIONS,
)

In [11]:
#### Have a conversation 
#### Starting conversation

In [12]:
query1 = "Hello, My name is Urula and I love Masala Chai"
response = await food_agent.run(query1, thread=thread)
print(response.text)

Hey Urula! Want a quick Masala Chai recipe or a twist on your favorite? Let me know what you’d like.


In [13]:
query2 = "I love spicier tea"
response2 = await food_agent.run(query2, thread=thread)
print(response2.text)

**Spicier Masala Chai**

| Ingredient | Amount |
|------------|--------|
| Water | 1 cup |
| Black tea leaves (e.g., Assam) | 2 tsp |
| Whole spices: cinnamon stick, 4 cardamom pods, 4 cloves |  |
| Fresh ginger | 1 inch, sliced |
| Crushed red pepper or cayenne | ½ tsp (adjust to taste) |
| Milk | ½ cup (or more for a latte) |
| Sweetener: honey, sugar, or maple syrup | 1–2 tsp |
| Optional: a pinch of black pepper or a drop of turmeric for extra zing |

### Quick Prep
1. **Boil** water with all spices, ginger, and black pepper for 4 min.
2. **Steep** tea leaves in the spiced water for 2 min.
3. **Strain** into a mug and add milk and sweetener.
4. **Heat** gently until warm (don’t boil once milk added).
5. **Serve** hot—extra spicy if you add a fresh pinch of red pepper or a dash of cayenne right before sipping.

Enjoy the extra kick! Let me know if you want a milder version or a chai latte twist.


In [14]:
# Show messages are stored in Redis
messages = await redis_store.list_messages()
print(f"\nTotal messages in Redis: {len(messages)}")


Total messages in Redis: 4


In [15]:
response3 = await food_agent.run("What do you know about me?", thread=thread)
print(response3.text)

I only know what’s been shared in our chat: your name is Urula and you love Masala Chai—especially the spicier versions. I don’t have any other personal info about you. If there’s anything else you’d like me to remember for future recipes, just let me know!


In [ ]:
# Cleanup
await redis_store.clear()
await redis_store.aclose()
print("Cleaned up Redis data\n")